Nessa versão foi alterado apenas o Alterando threshold

In [23]:
import pandas as pd
import pickle
import numpy as np

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)


In [24]:
# ============================================================
# 1. PARÂMETROS DO MLP
# ============================================================

MLP_PARAMS = {
    # Arquitetura da rede
    "hidden_layer_sizes": (64, 32, 16),

    # Função de ativação dos neurônios
    "activation": "relu",

    # Algoritmo de otimização
    "solver": "adam",

    # Taxa de aprendizado inicial
    "learning_rate_init": 0.001,

    # Quantidade máxima de épocas
    "max_iter": 100,

    # Tamanho do lote utilizado no treinamento
    "batch_size": 64,

    # Regularização L2
    "alpha": 0.0001,

    # Para interromper quando não houver melhoria
    "early_stopping": True,

    # Parte do treinamento utilizada para validação
    "validation_fraction": 0.1,

    # Número de épocas sem melhoria antes de parar
    "n_iter_no_change": 10,

    # Reprodutibilidade
    "random_state": 42
}

THRESHOLD = 0.90

In [25]:
# ============================================================
# 2. CARREGAMENTO DO DATASET
# ============================================================

arquivo = "C:/Users/gques/Documents/SPTECH_CODIGOS/ultimo-ano/tcc/backend/tabela_final/part-00000-13fcc452-2493-4299-ab17-d369f64d2ad3-c000.csv"


df = pd.read_csv(arquivo)

In [26]:
# ============================================================
# 3. VARIÁVEL TARGET
# ============================================================

target = "delivered_on_time"

# ============================================================
# 4. SPLIT TEMPORAL
# ============================================================

split_date = pd.to_datetime("2018-07-01", utc=True)

date_col = pd.to_datetime(
    df["order_delivered_carrier_date"],
    utc=True
)

train_mask = date_col < split_date
test_mask = date_col >= split_date

df_train = df[train_mask].copy()
df_test = df[test_mask].copy()

print(f"Treino: {len(df_train):,} linhas")
print(f"Teste: {len(df_test):,} linhas")

# ============================================================
# 5. REMOÇÃO DE COLUNAS COM POSSÍVEL DATA LEAKAGE
# ============================================================

colunas_remover = [
    target,
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "interval_code_delivered_carrier",
    "order_delivered_customer_date",
    "interval_code_delivered_customer",
    "order_estimated_delivery_date",
    "shipping_limit_date",
    "is_holiday_in_7_days",
    "is_holiday_in_14_days",
    "had_holiday_7_days_ago",
    "had_holiday_14_days_ago",
    "customer_city",
    "seller_city",
    "delivered_on_time", 
    "category_name",
    "review_score",
    "delivery_time_days"
]

colunas_remover = [
    coluna for coluna in colunas_remover
    if coluna in df.columns
]

Treino: 28,706 linhas
Teste: 5,114 linhas


In [27]:
# ============================================================
# 6. SEPARAÇÃO ENTRE X E Y
# ============================================================

X_train = df_train.drop(columns=colunas_remover)
y_train = df_train[target]

X_test = df_test.drop(columns=colunas_remover)
y_test = df_test[target]


# ============================================================
# 7. CONVERSÃO DE VARIÁVEIS CATEGÓRICAS
# ============================================================

X_train = pd.get_dummies(
    X_train,
    drop_first=True
)

X_test = pd.get_dummies(
    X_test,
    drop_first=True
)

X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

In [28]:
# ============================================================
# 8. TRATAMENTO DE VALORES AUSENTES
# ============================================================

X_train = X_train.replace(
    [np.inf, -np.inf],
    np.nan
)

X_test = X_test.replace(
    [np.inf, -np.inf],
    np.nan
)

X_train = X_train.fillna(
    X_train.median(numeric_only=True)
)

X_test = X_test.fillna(
    X_train.median(numeric_only=True)
)

X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

In [29]:
# ============================================================
# 9. NORMALIZAÇÃO
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [30]:
# ============================================================
# 10. CRIAÇÃO DO MLP
# ============================================================

mlp = MLPClassifier(
    **MLP_PARAMS
)


# ============================================================
# 11. TREINAMENTO
# ============================================================

mlp.fit(
    X_train_scaled,
    y_train
)

# ============================================================
# 11.1 SALVAMENTO DO MODELO
# ============================================================

modelo = {
    "model": mlp,
    "scaler": scaler,
    "features": X_train.columns.tolist(),
    "threshold": THRESHOLD
}

caminho_modelo = "C:/Users/gques/Documents/SPTECH_CODIGOS/ultimo-ano/tcc/backend/modelo/dl/mlp_model_v2.pkl"

with open(caminho_modelo, "wb") as arquivo:
    pickle.dump(modelo, arquivo)

print(f"\nModelo salvo em: {caminho_modelo}")


Modelo salvo em: C:/Users/gques/Documents/SPTECH_CODIGOS/ultimo-ano/tcc/backend/modelo/dl/mlp_model_v2.pkl


In [31]:
# ============================================================
# 12. PREDIÇÃO
# ============================================================

#VALORES DE PREDIÇÃO UTILIZADOS NO V1
# y_pred = mlp.predict(
#     X_test_scaled
# )

# y_prob = mlp.predict_proba(
#     X_test_scaled
# )[:, 1]

y_prob = mlp.predict_proba(
    X_test_scaled
)[:, 1]

y_pred = (y_prob >= THRESHOLD).astype(int)

In [32]:
# ============================================================
# 13. MÉTRICAS
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    pos_label=0,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    pos_label=0,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    pos_label=0,
    zero_division=0
)

f2 = fbeta_score(
    y_test,
    y_pred,
    beta=2,
    pos_label=0,
    zero_division=0
)

auc = roc_auc_score(
    y_test,
    y_prob
)

In [33]:
# ============================================================
# 14. RESULTADOS
# ============================================================

print("\n" + "=" * 60)
print("RESULTADOS - MLP")
print("=" * 60)

print(f"Arquitetura: {MLP_PARAMS['hidden_layer_sizes']}")
print(f"Activation: {MLP_PARAMS['activation']}")
print(f"Learning rate: {MLP_PARAMS['learning_rate_init']}")
print(f"Batch size: {MLP_PARAMS['batch_size']}")
print(f"Épocas máximas: {MLP_PARAMS['max_iter']}")

print("\nQuantidade de amostras:")
print(f"Treino: {len(X_train)}")
print(f"Teste:  {len(X_test)}")

print("\nMétricas:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"F2-score:  {f2:.4f}")
print(f"ROC-AUC:   {auc:.4f}")

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred))

print("\nRelatório de classificação:")
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)

print("\nQuantidade de épocas executadas:")
print(mlp.n_iter_)


RESULTADOS - MLP
Arquitetura: (64, 32, 16)
Activation: relu
Learning rate: 0.001
Batch size: 64
Épocas máximas: 100

Quantidade de amostras:
Treino: 28706
Teste:  5114

Métricas:
Accuracy:  0.8604
Precision: 0.1314
Recall:    0.0294
F1-score:  0.0480
F2-score:  0.0348
ROC-AUC:   0.4735

Matriz de confusão:
[[  18  595]
 [ 119 4382]]

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.13      0.03      0.05       613
           1       0.88      0.97      0.92      4501

    accuracy                           0.86      5114
   macro avg       0.51      0.50      0.49      5114
weighted avg       0.79      0.86      0.82      5114


Quantidade de épocas executadas:
21


# Análise do efeito do Threshold no MLP

O threshold teve efeito, mas o resultado mostra uma coisa importante: ele melhorou a detecção de atrasos, mas o MLP continua com problema de separação entre as classes.

## Comparação dos resultados

| Métrica | Antes | Agora |
|---|---:|---:|
| Accuracy | 87,74% | 86,04% |
| Precision atraso | 6,25% | 13,14% |
| Recall atraso | 0,16% | 2,94% |
| F1 atraso | 0,32% | 4,80% |
| F2 atraso | 0,20% | 3,48% |
| ROC-AUC | 0,4735 | 0,4735 |

## O que mudou na prática

A matriz de confusão anterior era:

```text
[[   1  612]
 [  15 4486]]

Agora:

```text
[[  18  595]
 [ 119 4382]]
```

Considerando:

- `0` = atraso
- `1` = no prazo

O modelo agora encontrou:

**18 dos 613 atrasos.**

Antes encontrava apenas:

**1 dos 613 atrasos.**

Então o recall subiu de:

```text
0,16% → 2,94%
```

Isso mostra que **o threshold realmente alterou o comportamento do classificador**.

Por outro lado, ele também passou a classificar **119 entregas que estavam no prazo como atrasadas**.

## O ponto mais importante: ROC-AUC

Você continua com:

```text
ROC-AUC: 0.4735
```

Isso é importante porque o ROC-AUC **não depende do threshold escolhido**.

Então:

```text
Threshold 0.50
      ↓
ROC-AUC = 0.4735
```

```text
Threshold 0.90
      ↓
ROC-AUC = 0.4735
```

Isso mostra que o problema não está simplesmente no threshold.

O modelo está produzindo probabilidades que **não estão separando bem atraso vs. no prazo**.

Em outras palavras, o threshold conseguiu alterar a quantidade de exemplos classificados como atraso, aumentando o recall. Porém, o ROC-AUC permaneceu em **0,4735**, indicando que o MLP continua apresentando pouca capacidade de separar as duas classes.